In [65]:
import os
import re
import glob
import copy
import enum
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Tuple, Dict, Optional, List

plt.close("all")

In [66]:
input_folder = "../build/rootNodeSolve__2023.11.27_11-05-36"

In [67]:

class RunOptions(enum.Enum):
    Baseline = "baseline"
    MIR = "mir"
    SplitGready = "split,gready"
    SplitFract = "split,fract"
    Pureint = "pureint"
    Hermite = "hermite"

class GroupType(enum.Enum):
    TotalTime = "total_time"
    CutGenerationTime = "cuts_time"
    BoostingTime = "boosting_time"
    IsIncumbentFound = "incumbent_found"
    SolutionStatus = "solution_status"
    ILPSolutionValue = "ilp_value"
    DualityGapDecrease = "duality_gap_decrease"
    DualityGapDecreaseLp = "duality_gap_decrease_lp"


name_pattern = r"(?<=/)[^/]+(?=.mps.gz_)"
run_option_pattern = r"\[(.*?)\]"
total_time_pattern = r"Total \(root\+branch&cut\) =\s*(?P<" + GroupType.TotalTime.value + r">\d+\.\d+)"
time_cuts_pattern = r"Total cutting plane generation time: (?P<" + GroupType.CutGenerationTime.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
time_boosting_pattern = r"Total boosting time: (?P<" + GroupType.BoostingTime.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
duality_gap_pattern = r"Duality gap decrease \(ILP\): (?P<" + GroupType.DualityGapDecrease.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
ilp_value_cuts_pattern = r"\nILP solution value:\s*(?P<" + GroupType.ILPSolutionValue.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
incumbent_pattern = r"An incumbent is found"
solution_status_pattern = r"Solution status is (?P<" + GroupType.SolutionStatus.value + r">\w+)"
lp_duality_gap_pattern = r"Duality gap decrease \(LP\): (?P<" + GroupType.DualityGapDecreaseLp.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"

In [68]:
def search_group(pattern: str, source: str, group_name: str) -> Optional[str]:
    match = re.search(pattern, source)
    if match:
        found = match.groupdict()[group_name]
        return found
    return None


def parse_file(filename: str) -> Dict[str, str]:
    results = {}
    
    with open(filename, "r") as file:
        file_content = file.read()
        total_time = 0.0
        time_cuts = 0.0
        time_boosting = 0.0
        is_incumbent_found = False
        solution_status = ""
        ilp_value = 0.0
        duality_gap = 0.0
        duality_gap_lp = 0.0

        if "terminate..." not in file_content:
            found_total_time = search_group(total_time_pattern, file_content, GroupType.TotalTime.value)
            total_time = found_total_time if found_total_time else 0.0
            
            found_time_cuts = search_group(time_cuts_pattern, file_content, GroupType.CutGenerationTime.value)
            time_cuts = found_time_cuts if found_time_cuts else 0.0
            
            found_time_boosting = search_group(time_boosting_pattern, file_content, GroupType.BoostingTime.value)
            time_boosting = found_time_boosting if found_time_boosting else 0.0

            match = re.search(incumbent_pattern, file_content)
            if match:
                is_incumbent_found = True

            found_solution_status = search_group(solution_status_pattern, file_content, GroupType.SolutionStatus.value)
            solution_status = found_solution_status if found_solution_status else False

            found_ilp_value = search_group(ilp_value_cuts_pattern, file_content, GroupType.ILPSolutionValue.value)
            ilp_value = found_ilp_value if found_ilp_value else 0.0

            found_duality_gap = search_group(duality_gap_pattern, file_content, GroupType.DualityGapDecrease.value)
            duality_gap = found_duality_gap if found_duality_gap else 0.0

            found_duality_gap_lp = search_group(lp_duality_gap_pattern, file_content, GroupType.DualityGapDecreaseLp.value)
            duality_gap_lp = found_duality_gap_lp if found_duality_gap_lp else 0.0
        
        results[GroupType.TotalTime.value] = total_time
        results[GroupType.CutGenerationTime.value] = time_cuts
        results[GroupType.BoostingTime.value] = time_boosting
        results[GroupType.IsIncumbentFound.value] = is_incumbent_found
        results[GroupType.SolutionStatus.value] = solution_status
        results[GroupType.ILPSolutionValue.value] = ilp_value
        results[GroupType.DualityGapDecrease.value] = duality_gap
        results[GroupType.DualityGapDecreaseLp.value] = duality_gap_lp

    return results

In [69]:
def construct_dataframe(data: dict) -> pd.DataFrame:
    return pd.DataFrame.from_dict(
        {problem: list(runOptions.values()) for problem, runOptions in data.items()},
        orient='index',
        columns=[option.value for option in RunOptions]
    )

# creating data frame with pure data
def parse_logs(logs_path: str) -> Tuple[pd.DataFrame]:
    
    problem_names: set = set()
    time_data = {}
    cut_generation_data = {}
    boosting_time_data = {}
    incumbent_data = {}
    solution_status_data = {}
    ilp_solution_value_data = {}
    duality_gap_data = {}
    duality_gap_lp_data = {}
    
    data_dicts = [time_data, cut_generation_data, boosting_time_data, incumbent_data,
                  solution_status_data, ilp_solution_value_data, duality_gap_data, duality_gap_lp_data]


    for filename in glob.glob(f"{logs_path}/*__log.txt"):
        problem_name = re.search(name_pattern, filename)[0]
        run_option = re.search(run_option_pattern, filename)[1]

        if problem_name not in problem_names:
            problem_names.add(problem_name)
            for data in data_dicts:
                data[problem_name] = {option.value: [] for option in RunOptions}
        
        problem_run_data : Dict[str, float] = parse_file(filename)
        for data, group_name in zip(data_dicts, [group.value for group in GroupType]):
            data[problem_name][run_option] = problem_run_data[group_name]
    
    time_df = construct_dataframe(time_data).apply(pd.to_numeric).dropna(axis=1, how='all')
    columns = time_df.columns
    cut_time_df = construct_dataframe(cut_generation_data).apply(pd.to_numeric)[columns]
    boosting_time_df = construct_dataframe(boosting_time_data).apply(pd.to_numeric)[columns]
    incumbent_df = construct_dataframe(incumbent_data)[columns]
    solution_status_df = construct_dataframe(solution_status_data)[columns]
    ilp_df = construct_dataframe(ilp_solution_value_data).apply(pd.to_numeric)[columns]
    duality_gap_df = construct_dataframe(duality_gap_data).apply(pd.to_numeric)[columns]
    duality_gap_lp_data = construct_dataframe(duality_gap_lp_data).apply(pd.to_numeric)[columns]

    return time_df, cut_time_df, boosting_time_df, incumbent_df, solution_status_df, ilp_df, duality_gap_df, duality_gap_lp_data

time_df, cut_time_df, boosting_time_df, incumbent_df, solution_status_df, ilp_df, duality_gap_df, duality_gap_lp_data = parse_logs(input_folder)

In [70]:
time_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,0.03,0.01,0.01,0.01
neos859080,0.26,0.49,0.15,0.25


In [71]:
cut_time_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,0,1.132290e+09,4.151070e+09,4.040670e+09
neos859080,0,1.462070e+08,5.957630e+08,5.152970e+08


In [72]:
boosting_time_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,0,1.160080e+09,4.180150e+09,4.069880e+09
neos859080,0,1.725420e+08,6.248890e+08,5.495920e+08


In [73]:
incumbent_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,True,True,True,True
neos859080,False,False,False,False


In [74]:
solution_status_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,Optimal,Optimal,Optimal,Optimal
neos859080,Infeasible,Infeasible,Infeasible,Infeasible


In [75]:
ilp_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,37.0,37.0,37.0,37.0
neos859080,0.0,0.0,0.0,0.0


In [76]:
duality_gap_df

,baseline,mir,"split,gready","split,fract"
enlight_hard,0.0,0.418919,0.418919,0.418919
neos859080,0.0,0.000000,0.000000,0.000000


In [77]:
duality_gap_lp_data

,baseline,mir,"split,gready","split,fract"
enlight_hard,0.0,0.418919,4.189190e-01,4.189190e-01
neos859080,0.0,0.000000,4.218850e-90,4.218850e-90
